# Assignment


## Brief

Write the Python codes for the following questions.


## Instructions

- Paste the answer as Python in the answer code section below each question.


### Connections

In [3]:
import os

from dotenv import load_dotenv
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi

In [4]:
# Load environment variables from .env file
load_dotenv()
MONGODB_URI = os.getenv('MONGODB_URI')
if not MONGODB_URI:
    raise ValueError(
        "❌ MONGODB_URI not found!\n"
        "Please create a .env file with your MongoDB credentials.\n"
        "See README.md for setup instructions."
    )
client = MongoClient(MONGODB_URI, server_api=ServerApi('1'))
# Send a ping to confirm a successful connection
try:
    client.admin.command('ping')
    print("✅ Successfully connected to MongoDB!")
except Exception as e:
    print(e)

✅ Successfully connected to MongoDB!


In [5]:
db = client.sample_mflix
movies = db.movies
print(f"📊 Database: {db.name}")
print(f"📁 Collection: movies ({movies.count_documents({})} documents)")

📊 Database: sample_mflix
📁 Collection: movies (21349 documents)



### Question 1

Question: From the `movies` collection, return the documents with the `plot` that starts with `"war"` in acending order of released date, print only title, plot and released fields. Limit the result to 5.


**Old Solution:**

In [ ]:
## Old Solution 

for m in movies.find({"plot": {"$regex": "war"}}).sort('released', pymongo.DESCENDING).limit(5):
    print(f"{m['title']} ({m['plot']}) was released in {m['released']}")

**Solution:**

In [9]:
# ============================================================================
# Question 1
# ============================================================================

results = []
for m in movies.find({"plot": {"$regex": "^war", "$options": "i"}}).sort('released', pymongo.ASCENDING).limit(5):
        results.append({
            'title': m['title'],
            'released': m['released'],
            'plot': m['plot']
        })

print("📽️  Movies with plot starting with 'War':")  
for r in results:
    print(f"🎬 {r['title']} ({r['released'].year}): {r['plot'][:60]}...")

📽️  Movies with plot starting with 'War':
🎬 Nausicaè of the Valley of the Wind (1984): Warrior/pacifist Princess Nausicaè desperately struggles to ...
🎬 Nausicaè of the Valley of the Wind (1984): Warrior/pacifist Princess Nausicaè desperately struggles to ...
🎬 Heaven and Earth (1991): Warlords Kagetora and Takeda each wish to prevent the other ...
🎬 Under the Stars (2007): Warning! This synopsis contains spoilers Bajo las estrellas ...
🎬 Aliens vs. Predator: Requiem (2007): Warring alien and predator races descend on a small town, wh...


### Question 2

Question: Group by `rated` and count the number of movies in each.

**Answer:**

In [12]:
# ============================================================================
# Question 2
# ============================================================================
stage_group_rated = {
    "$group": {
    "_id": "$rated",
    "movie_count": { "$sum": 1 },
    }
}

pipeline = [stage_group_rated,]
results = list(movies.aggregate(pipeline))

# Print the results
print("📊 Movie count by rating:")
for r in results:
    print(f"🎬 {r['_id']}: {r['movie_count']}")

📊 Movie count by rating:
🎬 TV-MA: 60
🎬 PG-13: 2321
🎬 TV-14: 89
🎬 TV-G: 59
🎬 Not Rated: 1
🎬 None: 9894
🎬 AO: 3
🎬 GP: 44
🎬 OPEN: 1
🎬 G: 477
🎬 R: 5537
🎬 APPROVED: 709
🎬 M: 37
🎬 TV-PG: 76
🎬 PG: 1852
🎬 PASSED: 181
🎬 Approved: 5
🎬 TV-Y7: 3



### Question 3

Question: Count the number of movies with 3 comments or more.


**Old Solution:**


In [ ]:
stage_lookup_comments = {
   "$lookup": {
         "from": "comments",
         "localField": "_id",
         "foreignField": "movie_id",
         "as": "related_comments",
   }
}

stage_add_comment_count = {
   "$addFields": {
         "comment_count": {
            "$size": "$related_comments"
         }
   }
}

stage_match_with_comments = {
   "$match": {
         "comment_count": {
            "$gte": 3
         }
   }
}

stage_group_count = {
   "$group": {
         "_id": None,
         "movie_count": { "$sum": 1 },
   }
}

pipeline = [
   stage_lookup_comments,
   stage_add_comment_count,
   stage_match_with_comments,
   stage_group_count,
]
results = movies.aggregate(pipeline)

for result in results:
   print(result)

**Solution:**

In [14]:
# ============================================================================
# Question 3
# ============================================================================
stage_match_with_comments = {
   "$match": {
         "num_mflix_comments": {
            "$gte": 3
         }
   }
}

stage_count_movies = {
   "$count": "number_movies_with_3_comments_or_more"
   }

pipeline = [
   stage_match_with_comments,
   stage_count_movies,
]
results = movies.aggregate(pipeline)
count = list(results)[0]['number_movies_with_3_comments_or_more']
print(f"🎬 Number of movies with 3 or more comments: {count}")  

🎬 Number of movies with 3 or more comments: 385


Extra Tips to alternative:

In [ ]:
count = movies_collection.count_documents({"num_mflix_comments": {"$gte": 3}})